In [1]:
import os
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [2]:
print(os.getcwd())

/Users/bettyk/Desktop/Risk_Factor_Analysis_of_Obstructive-Sleep_Apnea/10_22_Betty


In [3]:
os.listdir()

['age_over_80_logistic_regression.csv',
 'female_logistic_regression.csv',
 'male_logistic_regression.csv',
 'Female_logistic_regression.ipynb',
 'Total_logistic_regression.ipynb',
 'age_over_80_logistic_regression.ipynb',
 'age_between_65_80_logistic_regression.ipynb',
 '.ipynb_checkpoints',
 'age_between_65_80_logistic_regression.csv',
 'Male_logistic_regression.ipynb',
 'total_logistic_regression.csv']

In [4]:
# Step 1. 載入基本套件

# 讀取資料
df = pd.read_csv("../raw_data/age_over_80.csv")

# 查看前幾列資料，確定有成功載入
print(df.head())
print()

# 檢查欄位名稱、資料型態、缺漏值
print(df.info())
print()
print(df.isnull().sum())

   Second_Stroke  age  sex  HLOS  NIHSS  tPA(0/1)  EVT(0/1)  HTN(0/1)  \
0              0   91    1    18     32         0         0         1   
1              0   86    1    15      0         0         0         1   
2              0   84    1    31      2         0         0         1   
3              1   81    1     5     12         0         0         0   
4              1   87    1     9     16         0         0         1   

   DM(0/1)  Dyslipidemia(0/1)  Af(0/1)  smoking(Y/N/Q)   LDL   cholesterol  \
0        1                  0        0               0   78.2          127   
1        0                  0        1               0   83.6          161   
2        0                  0        1               0   71.2          168   
3        1                  1        0               0   72.4          133   
4        0                  1        0               0  102.1          191   

    TG   Cre  SGPT  HbA1c  MRS  
0  112  0.39    22    6.5    5  
1   75  0.49    14    5.7 

In [5]:
# 清理欄位名稱（避免空白、括號、特殊符號）
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
df.columns = df.columns.str.replace(r'_+$', '', regex=True)


# 把 N/Q 都合併成 0（非吸菸），Y=1
# 假設 smoking 欄位是 0=不吸菸, 1=吸菸, 2=戒菸
if 'smoking_Y_N_Q_' in df.columns:
    df['smoking_Y_N_Q_'] = df['smoking_Y_N_Q_'].replace({2: 0})
# print(df['smoking_Y_N_Q_'].value_counts())

# 目標變數 (Y)
Y = df['Second_Stroke']

# 自變數 (X) - 把你要放進模型的欄位都列出來
X = df.drop(columns=['Second_Stroke'])

# 只保留數值型欄位（防止類別文字造成報錯）
X = X.select_dtypes(include=[np.number])

# 移除缺值（Y + X 一起）
d = pd.concat([Y, X], axis=1).dropna()
Y = d['Second_Stroke']
X = d.drop(columns=['Second_Stroke'])

# 增加截距項 (Intercept)
X = sm.add_constant(X)

# 建立 Logistic Regression 模型
logit_model = sm.Logit(Y, X)
result = logit_model.fit(disp=False)

# 輸出結果摘要
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:          Second_Stroke   No. Observations:                  255
Model:                          Logit   Df Residuals:                      236
Method:                           MLE   Df Model:                           18
Date:                Mon, 06 Oct 2025   Pseudo R-squ.:                  0.2620
Time:                        20:58:06   Log-Likelihood:                -28.735
converged:                      False   LL-Null:                       -38.936
Covariance Type:            nonrobust   LLR p-value:                    0.3107
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -7.7845      8.980     -0.867      0.386     -25.384       9.815
age                  0.1185      0.109      1.088      0.277      -0.095       0.332
sex                 -1.0982 

/opt/anaconda3/envs/py310/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [6]:
print(df.columns.tolist())


['Second_Stroke', 'age', 'sex', 'HLOS', 'NIHSS', 'tPA_0_1', 'EVT_0_1', 'HTN_0_1', 'DM_0_1', 'Dyslipidemia_0_1', 'Af_0_1', 'smoking_Y_N_Q', 'LDL', 'cholesterol', 'TG', 'Cre', 'SGPT', 'HbA1c', 'MRS']


In [7]:
print([c for c in X.columns if 'smoking' in c.lower()])


['smoking_Y_N_Q']


In [8]:
# print(result.params)

In [9]:
# print(result.pvalues)

In [10]:
# print(result.conf_int())

In [11]:
# ==== 3. 整理成 Table（Adjusted OR, 95% CI, P value） ====
# 丟掉截距
params = result.params.drop(labels=["const"], errors="ignore")
pvals  = result.pvalues.drop(labels=["const"], errors="ignore")
conf   = result.conf_int().rename(columns={0:"lower",1:"upper"}).drop(labels=["const"], errors="ignore")

In [12]:
# print(result.params)

In [13]:
# 轉成 OR 與 95% CI (OR)
OR   = np.exp(params)
CI_l = np.exp(conf["lower"])
CI_u = np.exp(conf["upper"])

/opt/anaconda3/envs/py310/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [14]:
# 排序照原本欄位順序
order = [c for c in d.columns if c != "Second_Stroke" and c in params.index]
table = pd.DataFrame({
    "Adjusted odd's ratio": OR.reindex(order),
    "95 % Confidence Interval": [f"{CI_l.reindex(order)[i]:.2f}-{CI_u.reindex(order)[i]:.2f}" for i in order],
    "P value": [("<0.01" if pvals.reindex(order)[i] < 0.01 else f"{pvals.reindex(order)[i]:.2f}") for i in order]
}, index=order)

In [15]:
# 顯著性星號（p<0.05）
stars = pvals.reindex(order).apply(lambda p: "*" if p < 0.05 else "")
table["P value"] = table["P value"] + stars

In [16]:
# OR 四捨五入到兩位（看你期刊格式調整）
table["Adjusted odd's ratio"] = table["Adjusted odd's ratio"].round(2)

In [17]:
# # ==== 4.（可選）把變數名稱換成好看的顯示文字 ====

# # 建立一個 mapping dictionary：左邊是原始欄位名，右邊是報表要顯示的名稱
# pretty_names = {
#     'age': 'Age (years)',
#     'HLOS': 'Hospital length of stay (days)',
#     'NIHSS': 'NIH Stroke Scale score',
#     'tPA_0_1_': 'tPA (tissue Plasminogen Activator) treatment (0=No, 1=Yes)',
#     'EVT_0_1_': 'EVT (Endovascular thrombectomy)(0/1)',
#     'HTN_0_1_': 'Hypertension (0/1)',
#     'DM_0_1_': 'Diabetes mellitus (0/1)',
#     'Dyslipidemia_0_1_': 'Dyslipidemia (0/1)',
#     'Af_0_1_': 'Atrial fibrillation (0/1)',
#     'smoking_Y_N_Q_': 'Smoking (Y/N/Q)',
#     'LDL': 'LDL (mg/dL)',
#     'cholesterol': 'Total cholesterol (mg/dL)',
#     'TG': 'Triglycerides (mg/dL)',
#     'Cre': 'Creatinine (mg/dL)',
#     'SGPT': 'ALT (SGPT, U/L)',
#     'HbA1c': 'HbA1c (%)',
#     'MRS': 'Modified Rankin Scale at discharge'
# }

# # 使用 pandas 的 rename()，將索引(index) 改成好看的名字
# table = table.rename(index=pretty_names)

# 顯示表格
print("\n=== Multivariate logistic regression (Adjusted) ===")
display(table)

# （可選）存成 CSV，直接貼到論文表格
table.to_csv("age_over_80_logistic_regression.csv", encoding="utf-8-sig")


=== Multivariate logistic regression (Adjusted) ===


,Adjusted odd's ratio,95 % Confidence Interval,P value
age,1.13,0.91-1.39,0.28
sex,0.33,0.05-2.20,0.25
HLOS,1.03,0.97-1.09,0.31
NIHSS,1.09,0.96-1.24,0.19
tPA_0_1,0.00,0.00-inf,0.99
EVT_0_1,6.67,0.22-206.62,0.28
HTN_0_1,1.01,0.09-10.91,1.00
DM_0_1,1.40,0.21-9.51,0.73
Dyslipidemia_0_1,9.61,1.06-86.89,0.04*
Af_0_1,1.28,0.18-9.04,0.80
